In [10]:
import sqlite3
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv("Downloads/merged_master.csv")
print("Dataset Loaded Successfully!")

df.head()

Dataset Loaded Successfully!



,OrderID,OrderDate,CustomerID,CustomerName,Region,Product,Category,Qty,UnitPrice,Revenue,Month,PaymentMode
0,1001,2024-11-23,C115,Customer_C115,South,Laptop,Electronics,5,52193.81,260969.05,2024-11,UPI
1,1002,2024-12-12,C170,Customer_C170,North,Phone,Electronics,7,27190.70,190334.90,2024-12,Card
2,1003,2024-04-29,C165,Customer_C165,West,Laptop,Electronics,4,57376.22,229504.88,2024-04,Card
3,1004,2024-08-17,C176,Customer_C176,South,Monitor,Electronics,1,12621.14,12621.14,2024-08,NetBanking
4,1005,2024-06-23,C136,Customer_C136,East,Keyboard,Accessories,4,1637.16,6548.64,2024-06,UPI


In [3]:
engine = create_engine("sqlite:///sales.db")
print("Database Created Successfully!")


Database Created Successfully!


In [4]:
rows = df.to_sql(
    "sales",
    engine,
    if_exists="replace",
    index=False
)

print(f"{rows} rows inserted into SQL table.")


1000 rows inserted into SQL table.


In [11]:
conn = sqlite3.connect("sales.db")
cursor = conn.cursor()
cursor.execute("""

SELECT name FROM sqlite_master
WHERE type='table'

""")

print("Available Tables:")
print(cursor.fetchall())


Available Tables:
[('sales',)]


In [12]:
query = """

SELECT * FROM sales
LIMIT 5

"""

sample_df = pd.read_sql_query(query, engine)
print("Sample Data")
sample_df


Sample Data


,OrderID,OrderDate,CustomerID,CustomerName,Region,Product,Category,Qty,UnitPrice,Revenue,Month,PaymentMode
0,1001,2024-11-23,C115,Customer_C115,South,Laptop,Electronics,5,52193.81,260969.05,2024-11,UPI
1,1002,2024-12-12,C170,Customer_C170,North,Phone,Electronics,7,27190.70,190334.90,2024-12,Card
2,1003,2024-04-29,C165,Customer_C165,West,Laptop,Electronics,4,57376.22,229504.88,2024-04,Card
3,1004,2024-08-17,C176,Customer_C176,South,Monitor,Electronics,1,12621.14,12621.14,2024-08,NetBanking
4,1005,2024-06-23,C136,Customer_C136,East,Keyboard,Accessories,4,1637.16,6548.64,2024-06,UPI


In [13]:
query = """
SELECT OrderID, CustomerName, Revenue FROM sales
LIMIT 10
"""

select_df = pd.read_sql_query(query, engine)
print("SELECT Example")

select_df


SELECT Example


,OrderID,CustomerName,Revenue
0,1001,Customer_C115,260969.05
1,1002,Customer_C170,190334.90
2,1003,Customer_C165,229504.88
3,1004,Customer_C176,12621.14
4,1005,Customer_C136,6548.64
5,1006,Customer_C149,192509.82
6,1007,Customer_C169,192309.60
7,1008,Customer_C125,30967.58
8,1009,Customer_C113,44580.70
9,1010,Customer_C146,4161.45


In [17]:
query = """
SELECT * FROM sales
WHERE Revenue > 5000
"""

where_df = pd.read_sql_query(query, engine)

print("WHERE Example")
where_df.head()


WHERE Example


,OrderID,OrderDate,CustomerID,CustomerName,Region,Product,Category,Qty,UnitPrice,Revenue,Month,PaymentMode
0,1001,2024-11-23,C115,Customer_C115,South,Laptop,Electronics,5,52193.81,260969.05,2024-11,UPI
1,1002,2024-12-12,C170,Customer_C170,North,Phone,Electronics,7,27190.70,190334.90,2024-12,Card
2,1003,2024-04-29,C165,Customer_C165,West,Laptop,Electronics,4,57376.22,229504.88,2024-04,Card
3,1004,2024-08-17,C176,Customer_C176,South,Monitor,Electronics,1,12621.14,12621.14,2024-08,NetBanking
4,1005,2024-06-23,C136,Customer_C136,East,Keyboard,Accessories,4,1637.16,6548.64,2024-06,UPI


In [19]:
query = """
SELECT Region, SUM(Revenue) AS TotalRevenue FROM sales
GROUP BY Region
"""

group_df = pd.read_sql_query(query, engine)
print("GROUP BY Example")
group_df


GROUP BY Example


,Region,TotalRevenue
0,East,19419533.12
1,North,17002359.39
2,South,18759074.14
3,West,19750378.66


In [21]:
query = """
SELECT Region, SUM(Revenue) AS TotalRevenue FROM sales
GROUP BY Region
HAVING SUM(Revenue) > 100000
"""

having_df = pd.read_sql_query(query, engine)
print("HAVING Example")
having_df


HAVING Example


,Region,TotalRevenue
0,East,19419533.12
1,North,17002359.39
2,South,18759074.14
3,West,19750378.66


In [23]:
query = """
SELECT CustomerName, Revenue FROM sales 
ORDER BY Revenue DESC
LIMIT 10
"""

order_df = pd.read_sql_query(query, engine)

print("ORDER BY Example")

order_df


ORDER BY Example


,CustomerName,Revenue
0,Customer_C171,475996.40
1,Customer_C104,470562.24
2,Customer_C120,459084.40
3,Customer_C133,458183.12
4,Customer_C124,452731.36
5,Customer_C132,428029.92
6,Customer_C179,420944.44
7,Customer_C150,417918.06
8,Customer_C175,411768.00
9,Customer_C168,411088.86


In [25]:
query1 = """

WITH ProductRevenue AS
(
SELECT Region, Product, SUM(Revenue) AS TotalRevenue
FROM sales
GROUP BY Region, Product
)

SELECT * FROM
(

SELECT Region, Product, TotalRevenue, RANK() OVER
(
PARTITION BY Region
ORDER BY TotalRevenue DESC
)

AS ProductRank FROM ProductRevenue
)

WHERE ProductRank <=5
ORDER BY Region, ProductRank
"""

top_products = pd.read_sql_query(query1, engine)
print("Top 5 Products by Revenue Per Region")
top_products


Top 5 Products by Revenue Per Region


,Region,Product,TotalRevenue,ProductRank
0,East,Laptop,6878615.18,1
1,East,Phone,5503746.30,2
2,East,Tablet,3672154.65,3
3,East,Monitor,1768819.10,4
4,East,Printer,827656.23,5
5,North,Laptop,6235568.11,1
6,North,Phone,4313389.76,2
7,North,Tablet,2626631.10,3
8,North,Monitor,2396103.33,4
9,North,Printer,855624.26,5


In [26]:
query2 = """

WITH MonthlyRevenue AS
(
SELECT Month, SUM(Revenue) AS Revenue FROM sales
GROUP BY Month
)

SELECT Month, Revenue, LAG(Revenue) OVER
(
ORDER BY Month
)
AS PreviousMonthRevenue, Revenue - LAG(Revenue) OVER
(
ORDER BY Month
)

AS RevenueChange FROM MonthlyRevenue
"""

monthly_df = pd.read_sql_query(query2, engine)
print("Month-over-Month Revenue Change")
monthly_df

Month-over-Month Revenue Change


,Month,Revenue,PreviousMonthRevenue,RevenueChange
0,2024-01,8169899.75,NaN,NaN
1,2024-02,5852632.31,8169899.75,-2317267.44
2,2024-03,6731316.05,5852632.31,878683.74
3,2024-04,4753853.45,6731316.05,-1977462.60
4,2024-05,5671884.24,4753853.45,918030.79
5,2024-06,5475152.11,5671884.24,-196732.13
6,2024-07,5922844.22,5475152.11,447692.11
7,2024-08,7004998.68,5922844.22,1082154.46
8,2024-09,6440421.16,7004998.68,-564577.52
9,2024-10,6257255.80,6440421.16,-183165.36


In [27]:
query3 = """
SELECT CustomerID, CustomerName, COUNT(OrderID) AS TotalOrders, AVG(Revenue) AS AverageOrderValue, SUM(Revenue) AS TotalRevenue
FROM sales
GROUP BY CustomerID, CustomerName
HAVING COUNT(OrderID) > 5 AND
AVG(Revenue) > 5000
ORDER BY TotalRevenue DESC
"""

customer_df = pd.read_sql_query(query3, engine)
print("Customers with >5 Orders and Avg Order > ₹5000")
customer_df


Customers with >5 Orders and Avg Order > ₹5000


,CustomerID,CustomerName,TotalOrders,AverageOrderValue,TotalRevenue
0,C166,Customer_C166,17,124460.930000,2115835.81
1,C171,Customer_C171,19,108278.068947,2057283.31
2,C144,Customer_C144,18,114268.528333,2056833.51
3,C150,Customer_C150,18,101864.174444,1833555.14
4,C145,Customer_C145,17,101408.885294,1723951.05
...,...,...,...,...,...
75,C151,Customer_C151,12,34472.195833,413666.35
76,C101,Customer_C101,8,43696.228750,349569.83
77,C172,Customer_C172,9,31833.441111,286500.97
78,C155,Customer_C155,8,29756.322500,238050.58


In [28]:
top_products.to_csv(
    "top5_products_by_region.csv",
    index=False
)
monthly_df.to_csv(
    "monthly_revenue_change.csv",
    index=False
)
customer_df.to_csv(
    "customer_report.csv",
    index=False
)
print("CSV Reports Saved Successfully!")

conn.close()
engine.dispose()
print("\nDatabase Connection Closed.")
print("\nProgram Completed Successfully!")

CSV Reports Saved Successfully!

Database Connection Closed.

Program Completed Successfully!
